# Solar Panel Detection
**NB 08** | Netherlands | Sentinel-2 | SegFormer-B2

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import numpy as np

import pygeovision as pgv
from pygeovision.inference.tiled import TiledInference
from pygeovision.models import get_model

client = pgv.PyGeoVision()
print(client)


In [ ]:
BBOX       = (4.8, 52.3, 5.0, 52.5)
DATE_RANGE = ('2024-04-01', '2024-09-30')
PROVIDERS  = ['planetary_computer']

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 10,
)
print(f"Found {len(results)} scenes")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")


In [ ]:
BANDS = ['B02', 'B03', 'B04', 'B08']

downloads = client.download(
    results[:1],
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = ['reproject:EPSG:32631', 'cog'],
)
scene_path = downloads[0].path if downloads and downloads[0].success else None
scl_cands  = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path   = str(scl_cands[0]) if scl_cands else None
if scene_path:
    d = downloads[0]
    print(f"Downloaded   : {scene_path}")
    print(f"Size         : {d.bytes_downloaded/1024/1024:.1f} MB")
    print(f"SCL mask     : {scl_path}")
else:
    print("Download failed or scene unavailable — check provider availability")


In [ ]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = 4,
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")


In [ ]:
PREPROCESSED = DATA_DIR / 'netherlands_preprocessed.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4,5,6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'detection',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")
    arr = ready['array']
    print(f"Value range         : {arr.min():.4f} → {arr.max():.4f}")
    print(f"NaN count           : {np.isnan(arr).sum()}")
else:
    print("No scene available — cannot preprocess")
    ready = None


In [ ]:
# NDBI (built-up) and EVI to distinguish solar panels from rooftops
if PREPROCESSED.exists():
    idx_dir = DATA_DIR / 'indices'; idx_dir.mkdir(exist_ok=True)
    ndbi = client.indices.ndbi(str(PREPROCESSED), swir1_band=4, nir_band=3,
        output_path=str(idx_dir/'ndbi.tif'))
    evi  = client.indices.evi(str(PREPROCESSED),  blue_band=1, red_band=3, nir_band=4,
        output_path=str(idx_dir/'evi.tif'))
    import rasterio
    for name, p in [('NDBI', ndbi), ('EVI', evi)]:
        with rasterio.open(p) as src: a = src.read(1)
        print(f"{name}: mean={a.mean():.3f}  built-up={( a>0).mean()*100:.1f}%")


In [ ]:
PREDICTION = DATA_DIR / 'solar_raw.tif'

if ready and PREPROCESSED.exists():
    model = get_model('segformer-b2', num_classes=2, pretrained=True)
    inf   = TiledInference(model, chip_size=512, overlap=64, blend_mode='gaussian')
    res   = inf.infer(str(PREPROCESSED), str(PREDICTION))
    print(f"Inference: {res.get('n_chips')} chips  {res.get('duration_seconds',0):.1f}s  "
          f"{res.get('chips_per_second',0):.1f} chips/s")
else:
    print("Skipping inference — no preprocessed scene")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=5, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=10.0, simplify_tolerance=0.2)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


In [ ]:
# Energy potential estimation from panel area
stats  = client.postprocess.class_statistics(str(DATA_DIR/'sieved.tif'))
panel_area_m2 = stats.get(1,{}).get('area_m2', 0)
panel_count   = len(json.load(open(DATA_DIR/'output.geojson'))['features'])
# Typical solar panel: ~1.7m² × ~400W, ~1200 kWh/year/kWp in Netherlands
kw_capacity   = panel_area_m2 / 1.7 * 0.4
annual_kwh    = kw_capacity * 1200
print(f"Detected panels      : {panel_count}")
print(f"Panel area           : {panel_area_m2:.0f} m²")
print(f"Estimated capacity   : {kw_capacity:.0f} kW")
print(f"Annual energy (est.) : {annual_kwh/1000:.0f} MWh")


**NB08 — Solar Detection** complete.
- Study area: Netherlands
- Sensor: Sentinel-2 10m
- Model: SegFormer-B2
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG